# Select and Configure Compute in Azure Databricks

## 🏥 Healthcare Industry Scenario

**MedTech Analytics Corporation** is a healthcare data analytics company that processes millions of patient records, lab results, and medical device telemetry data daily. Their Azure Databricks platform supports various workloads:

- **Real-time patient monitoring**: Streaming data from IoT medical devices
- **Clinical data warehousing**: ETL pipelines processing EHR (Electronic Health Records)
- **Medical research analytics**: Ad-hoc analysis by data scientists
- **Regulatory reporting**: Automated HIPAA compliance reports
- **Predictive healthcare models**: Machine learning for early diagnosis

This notebook demonstrates how to select and configure compute resources optimized for different healthcare data engineering workloads while maintaining security, cost efficiency, and performance.

**Setup**: Run the setup script to create sample healthcare data tables.

In [ ]:
from scripts.setup import setup, setup_02

spark.sql("USE CATALOG trainer_demo")
setup_02(spark)

## Choose an appropriate compute type

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/select-and-configure-compute.choose-appropriate-compute-type.png)

Different compute types suit different scenarios. The following table compares key characteristics to help you make informed decisions:

| Compute type                | Recommended for                                        | Startup time  | Management overhead                          | Cost efficiency                                       | Key limitation                        |
| --------------------------- | ------------------------------------------------------ | ------------- | -------------------------------------------- | ----------------------------------------------------- | ------------------------------------- |
| Serverless compute          | Interactive development, ETL jobs, BI workloads        | ⚡ 2-6 seconds | 👌 Minimal - fully managed                    | 🟢 High - scales to zero, pay only for usage           | No RDD APIs, R, or JAR libraries      |
| Classic compute (Standard)  | Collaborative data engineering, shared analytics       | ⏱️ 3-7 minutes | 🔧 Moderate - configure and monitor           | 🟡 Moderate - multi-user sharing reduces per-user cost | Requires Unity Catalog for governance |
| Classic compute (Dedicated) | RDD workloads, GPU jobs, R language, custom containers | ⏱️ 3-7 minutes | 🔧 Moderate - configure and monitor           | 🔴 Lower - single user/group only                      | Higher cost than shared resources     |
| Instance pools              | Frequently run classic workloads needing fast startup  | 🚀 <1 minute   | ⚠️ Higher - maintain idle capacity            | 🔄 Variable - justify idle cost with usage frequency   | Pay for idle instances                |
| SQL warehouse (Serverless)  | SQL analytics, BI dashboards, reporting                | ⚡ 2-6 seconds | 👌 Minimal - intelligent workload management  | 🟢 High - dynamic scaling with Photon + Predictive IO  | SQL workloads only                    |
| SQL warehouse (Pro)         | SQL with custom networking, federation                 | ⏱️ ~4 minutes  | 🔧 Moderate - manual scaling configuration    | 🟡 Moderate - Photon + Predictive IO                   | Slower scaling than serverless        |
| SQL warehouse (Classic)     | Entry-level SQL exploration                            | ⏱️ ~4 minutes  | 🔧 Moderate - manual scaling configuration    | 🔴 Lower - basic Photon only                           | Limited performance features          |
| Job compute (Serverless)    | Automated workflows, production ETL                    | ⚡ 2-6 seconds | 👌 Minimal - auto-terminated after completion | 🟢 High - no idle costs                                | Same as serverless compute            |
| Job compute (Classic)       | Jobs requiring custom configurations                   | ⏱️ 3-7 minutes | 🔧 Moderate - configure policies              | 🟡 Moderate - auto-termination prevents idle waste     | Requires infrastructure management    |

## Configure compute performance

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/select-and-configure-compute.configure-compute-performance.png)

![diagram](https://learn.microsoft.com/en-us/training/wwl-databricks/select-and-configure-compute/media/instance-pool-management.png)

### 🎯 DEMO: Node Type Selection for Healthcare Workloads

Different healthcare workloads benefit from different VM instance types based on their resource requirements.

**Healthcare Workload → Node Type Mapping:**

| Workload | Node Type | Reason |
|----------|-----------|--------|
| **Lab results aggregation** (Large joins) | **Memory-optimized (E-series)** | Joins millions of test results with patient records |
| **Medical device telemetry ETL** (Simple transforms) | **Compute-optimized (F-series)** | Straightforward transformations, high CPU usage |
| **Genomic data analysis** (Fast local I/O) | **Storage-optimized (L-series)** | Repeatedly scans large genomic sequence files |
| **Deep learning - Medical imaging** (GPU required) | **GPU instances (NC/ND-series)** | Training CNNs on X-ray/MRI images |

**Demo Setup:**
1. Navigate to **Compute** → **Create Compute**
2. Select appropriate instance type based on workload
3. Configure worker count and autoscaling

### 🎯 DEMO: Configuring Autoscaling

**Healthcare Use Case: Variable Load ETL Pipeline**

MedTech's hourly pipeline processes varying volumes of medical device data:
- **Off-peak (2-6 AM)**: 10K records/hour → Need 2 workers
- **Peak (8 AM - 5 PM)**: 500K records/hour → Need 8 workers
- **Weekends**: Minimal data → Need 1 worker

**Autoscaling Configuration:**
```
Min workers: 2
Max workers: 8
Autoscaling mode: Optimized
```

**Benefits:**
- Scales up when device data spikes
- Scales down during off-peak hours
- Reduces costs by 40-60% vs. fixed 8-worker cluster

### 🎯 DEMO: Automatic Termination Settings

**Prevent compute cost waste with auto-termination:**

**Configuration Examples:**

| Use Case | Termination Setting | Reason |
|----------|---------------------|--------|
| **Data scientist exploration** | 45 minutes | Allows time for analysis review between queries |
| **Shared team cluster** | 60 minutes | Longer timeout for collaborative work |
| **Job cluster** | After completion | Automatic (no manual setting needed) |
| **Production long-running** | Disabled | 24/7 availability for critical workloads |

**To configure:**
1. **Compute** → **Edit Cluster**
2. **Advanced Options** → **Automatic Termination**
3. Set minutes (e.g., 45)

### 🎯 DEMO: Spot Instances for Cost Optimization

**MedTech's Cost Savings Strategy:**

**Worker nodes**: Use **Spot instances** (Azure Spot VMs)
- Up to 90% cost reduction
- Spark handles interruptions automatically
- Decommissioning migrates data before termination

**Driver node**: Use **On-demand instances**
- Ensures cluster reliability
- Driver failure = entire cluster fails

**Configuration:**
```
Worker instances: Azure Spot VMs
Driver instance: On-demand
Enable decommissioning: Yes
```

**Ideal for:**
- Batch ETL jobs
- Non-time-critical analytics
- Development/testing workloads

**Avoid for:**
- Real-time patient monitoring
- SLA-critical applications
- Interactive executive dashboards

### 🎯 DEMO: Using Instance Pools

**When MedTech uses instance pools:**

Their data engineering team frequently creates/destroys clusters during development:
- 20 cluster creations per day
- Standard startup: 5 minutes
- With pool: <1 minute

**Pool Configuration:**
```
Min idle instances: 3
Max capacity: 10
Idle termination: 30 minutes
Preloaded runtime: 14.3 LTS
```

**Cost-Benefit:**
- Idle cost: 3 VMs × 24 hours = 72 VM-hours/day
- Savings: 20 clusters × 4 min saved = 80 minutes of productivity
- Justified when team creates ≥5 clusters daily

**To create a pool:**
1. **Compute** → **Instance Pools** → **Create**
2. Configure min/max instances
3. Select VM type
4. Preload runtime version
5. Create clusters from the pool

## Configure compute features

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/select-and-configure-compute.configure-compute-feature-settings.png)

### 🎯 DEMO: Photon Acceleration

**Photon** accelerates SQL and DataFrame operations using native code execution.

**When MedTech uses Photon:**
- ✅ Complex aggregations on millions of lab results
- ✅ Large joins between patient and billing tables
- ✅ Scanning wide tables with 100+ columns
- ❌ Simple transformations on small datasets (<1000 rows)
- ❌ GPU-based deep learning workloads

**To enable Photon:**
1. During cluster creation, expand **Performance** section
2. Toggle **Photon Acceleration** to **Enabled**
3. Select runtime version 9.1 LTS or above
4. **Note**: Photon is incompatible with GPU clusters

### 🎯 DEMO: Databricks Runtime Selection

**MedTech's Runtime Strategy:**

| Cluster Type | Runtime | Reason |
|--------------|---------|--------|
| **Development clusters** | Latest standard runtime | Access newest features and optimizations |
| **Production ETL jobs** | **LTS (Long-Term Support)** | Stability, extended compatibility, tested code |
| **ML training clusters** | **Databricks Runtime ML** | Pre-installed ML libraries (TensorFlow, PyTorch) |
| **Legacy pipelines** | Older LTS | Maintain compatibility until migration complete |

**Example Runtimes:**
- Standard: `14.3 LTS` (Apache Spark 3.5.0)
- ML: `14.3 LTS ML` (includes scikit-learn, TensorFlow, PyTorch, MLflow)

## Install libraries for compute

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/select-and-configure-compute.install-libraries-for-compute.png)

![diagram](https://learn.microsoft.com/en-us/training/wwl-databricks/select-and-configure-compute/media/library-installation.png)

### 🎯 DEMO: Installing Libraries for Healthcare Workloads

**MedTech's Library Requirements:**

| Use Case | Library | Installation Method |
|----------|---------|---------------------|
| **FHIR data parsing** | `fhir.resources` | PyPI |
| **SQL Server connectivity** | `pymssql` | PyPI (specific version) |
| **JDBC driver for SQL** | `mssql-jdbc` | Maven |
| **Custom HL7 parser** | Internal wheel file | Unity Catalog Volume |
| **Medical coding** | `icd10-cm` | PyPI |

**Installation Levels:**
- **Compute-scoped**: Available to all notebooks on cluster (persistent)
- **Notebook-scoped**: Available only to specific notebook (temporary)

### 🎯 DEMO: Install from PyPI

**Scenario**: MedTech needs to connect to SQL Server to query patient demographic data.

**Library**: `pymssql` (Python SQL Server driver)

**Steps:**
1. Navigate to **Compute** → Select your cluster
2. Click **Libraries** tab
3. Click **Install new**
4. Select **PyPI**
5. Enter package name: `pymssql==2.3.1` (pinned version for stability)
6. Click **Install**

**Best Practice**: Always specify version numbers for production workloads to ensure reproducibility.

### 🎯 DEMO: Install from Maven

**Scenario**: Connect to Azure SQL using JDBC for better performance.

**Library**: Microsoft JDBC Driver for SQL Server

**Maven Coordinates:**
```
com.microsoft.sqlserver:mssql-jdbc:12.8.1.jre11
```

**Steps:**
1. **Compute** → **Libraries** → **Install new**
2. Select **Maven**
3. Enter coordinates: `com.microsoft.sqlserver:mssql-jdbc:12.8.1.jre11`
4. Click **Install**

**Note**: Standard access mode clusters require Maven coordinates on the allowlist.

### 🎯 DEMO: Install from Unity Catalog Volumes

**Scenario**: MedTech developed a custom Python library for HL7 message parsing.

**Setup:**
1. Create a volume for shared libraries
2. Upload wheel file to volume
3. Install from volume path

**Advantages:**
- Unity Catalog permissions control who can install
- Audit logs track library usage
- Centralized management across workspaces

In [ ]:
%sql
-- Create a volume for storing custom libraries

CREATE VOLUME IF NOT EXISTS trainer_demo.demo_02.shared_libraries
  COMMENT 'Shared Python libraries for healthcare analytics';

-- Grant read access to data engineering team
-- GRANT READ VOLUME ON VOLUME trainer_demo.demo_02.shared_libraries TO `data_engineers`;

SELECT 'Volume created: /Volumes/trainer_demo/demo_02/shared_libraries' as status;

**Upload library to volume:**

1. Navigate to **Catalog Explorer**
2. Browse to `trainer_demo.demo_02.shared_libraries`
3. Click **Upload to this volume**
4. Upload `hl7_parser-1.2.3-py3-none-any.whl` (for this demo you can use an empty file)

**Install from volume:**

1. **Compute** → **Libraries** → **Install new**
2. Select **Python Whl**
3. Enter path: `/Volumes/trainer_demo/demo_02/shared_libraries/hl7_parser-1.2.3-py3-none-any.whl`
4. Click **Install**

**Security**: Standard access mode requires volume paths on allowlist.

### 🎯 DEMO: Using requirements.txt

**Scenario**: Install multiple healthcare analytics libraries at once.

**Create requirements.txt:**
```txt
pandas==2.1.0
numpy==1.24.3
fhir.resources==7.1.0
icd10-cm==0.0.4
sqlalchemy==2.0.21
```

**Upload to:**
- Workspace Files: `/Workspace/Users/you@example.com/requirements.txt`
- Unity Catalog Volume: `/Volumes/trainer_demo/demo_02/shared_libraries/requirements.txt`

**Install:**
1. **Compute** → **Libraries** → **Install new**
2. Select **Requirements**
3. Enter path to requirements.txt
4. Click **Install** → All packages install automatically

### 🎯 DEMO: Init Scripts (Advanced)

**Use init scripts for system-level configuration, NOT for library installation.**

**Valid Use Cases:**
- Install system packages (`apt-get install`)
- Configure environment variables
- Set up monitoring agents
- Install database drivers requiring system libraries

**Example: Configure healthcare data encryption settings**

```bash
#!/bin/bash
# File: /Volumes/trainer_demo/demo_02/scripts/security_init.sh

# Set HIPAA compliance environment variables
export ENABLE_ENCRYPTION=true
export AUDIT_LOGGING=enabled

# Configure TLS for database connections
echo "TLS_REQCERT=strict" >> /etc/openldap/ldap.conf

echo "Security configuration complete"
```

**Configure cluster to run init script:**
1. **Compute** → **Edit** → **Advanced options** → **Init Scripts**
2. Add path: `/Volumes/trainer_demo/demo_02/scripts/security_init.sh`
3. Standard access mode: Add path to allowlist first

### 🎯 DEMO: Standard Access Mode Allowlist

**For shared clusters with Standard access mode:**

**Metastore admin must allowlist:**
- Maven coordinates
- JAR file paths
- Init script paths

**Example allowlist entries:**
```
Maven: com.microsoft.sqlserver:mssql-jdbc
Volume path: /Volumes/trainer_demo/demo_02/shared_libraries/
Init scripts: /Volumes/trainer_demo/demo_02/scripts/
```

**To configure allowlist (admin only):**
1. **Catalog Explorer** → Select **Metastore**
2. **Settings** → **Allowed JARs/Init Scripts**
3. Click **Add**
4. Enter Maven coordinates or paths
5. Save

**Benefits:**
- Security team controls approved libraries
- Prevents unauthorized code execution
- Maintains audit trail
- Enables self-service within guardrails

## Configure compute access

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/select-and-configure-compute.configure-compute-access-perms.png)

### 🎯 DEMO: Compute Permission Levels

**MedTech's Permission Strategy:**

| Role | Permission Level | Can Do | Cannot Do |
|------|------------------|--------|-----------|
| **Data Analyst** | `CAN ATTACH TO` | Run queries, view Spark UI | Restart, edit config |
| **Senior Data Engineer** | `CAN RESTART` | Attach + restart clusters | Edit config, manage permissions |
| **Team Lead** | `CAN MANAGE` | Full control: edit, libraries, permissions | — |
| **No permissions** | None | — | Can't see the cluster |

**Permission Hierarchy:**
```
NO PERMISSIONS
    ↓
CAN ATTACH TO (view metrics, run notebooks)
    ↓
CAN RESTART (+ control cluster lifecycle)
    ↓
CAN MANAGE (+ edit config, install libraries, manage permissions)
```

### 🎯 DEMO: Configuring Compute Permissions

**Scenario**: Grant the **Healthcare Analytics team** access to the shared analysis cluster.

**Steps:**
1. Navigate to **Compute** → Select cluster
2. Click **Permissions** (three-dot menu or Permissions tab)
3. Click **Add**
4. Search for group: `healthcare_analysts`
5. Select permission: `CAN ATTACH TO`
6. Click **Save**

**Result**: All members of `healthcare_analysts` group can now:
- Attach notebooks to the cluster
- View cluster metrics and Spark UI
- Run queries and analysis
- BUT cannot restart or modify the cluster

**Best Practice: Use Groups, Not Individual Users**

❌ **Don't:**
```
alice@medtech.com → CAN ATTACH TO
bob@medtech.com → CAN ATTACH TO
charlie@medtech.com → CAN ATTACH TO
```

✅ **Do:**
```
healthcare_analysts (group) → CAN ATTACH TO
  ├─ alice@medtech.com
  ├─ bob@medtech.com
  └─ charlie@medtech.com
```

**Benefits:**
- Easier management as team changes
- Consistent permissions across resources
- Automatic access for new team members
- Simpler audit and compliance

### 🎯 DEMO: Workspace Entitlements

**Unrestricted Cluster Creation Entitlement:**

By default, only admins can create clusters. Grant this entitlement to allow non-admins to create clusters without size restrictions.

**MedTech Example:**
- Data Engineers: ✅ Unrestricted cluster creation
- Data Analysts: ❌ Use pre-configured shared clusters only
- Data Scientists: ✅ Unrestricted (for ML experimentation)

**To grant:**
1. **Settings** → **Identity and access** → **Users** or **Groups**
2. Select user/group
3. **Entitlements** → Enable **Allow cluster creation**
4. Save